# Asistente de consulta (RAG) · Café, papa y valor agregado agropecuario en Nariño

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/santacruzdavid1201-web/cafe-papa-narino/blob/main/rag/consulta_rag.ipynb)

Chat que responde preguntas sobre el proyecto **usando solo el corpus del repositorio**: el artículo en español y en inglés, la documentación, el texto metodológico del notebook, las tablas de resultados y las fichas de las referencias. Cada respuesta cita los fragmentos usados (documento, sección y página).

**Cómo usarlo**
1. Para respuestas redactadas: *Entorno de ejecución → Cambiar tipo de entorno → GPU T4* (gratuito). Sin GPU funciona en modo extractivo (devuelve las oraciones más pertinentes del corpus, con sus citas).
2. *Entorno de ejecución → Ejecutar todas*. La primera vez descarga los modelos abiertos (unos 3 minutos con GPU).
3. Escriba la pregunta en la caja de la sección 5, en español o en inglés.

**Componentes**
* Recuperación híbrida: BM25 (léxica) + embeddings multilingües `intfloat/multilingual-e5-small`, combinadas por fusión de rangos recíprocos.
* Generación: `Qwen/Qwen2.5-3B-Instruct` (licencia abierta) con instrucciones de responder solo con el contexto y citar cada afirmación.
* Evaluación: preguntas de control con la respuesta conocida; se reporta acierto@k y MRR de la recuperación.
* Para ampliar el corpus, copie archivos PDF en `rag/corpus_extra/` y vuelva a ejecutar.

## 1. Preparación

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
EN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/santacruzdavid1201-web/cafe-papa-narino"
candidatas = [Path.cwd().parent, Path.cwd(), Path("/content/cafe-papa-narino")]
BASE = next((p for p in candidatas if (p / "rag" / "config.json").exists()), None)
if BASE is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/cafe-papa-narino"], check=True)
    BASE = Path("/content/cafe-papa-narino")
if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "pypdf", "accelerate"], check=True)
sys.path.insert(0, str(BASE / "rag"))
from rag_nucleo import construir_corpus, trocear, Buscador, Generador, evaluar
print("Proyecto en:", BASE)

## 2. Configuración

In [ ]:
MODO_GENERADOR = "auto"      # "auto" (GPU → Qwen2.5-3B; CPU → extractivo), "qwen-3b", "qwen-0.5b" o "extractivo"
USAR_EMBEDDINGS = True       # False: solo BM25 (sin descargar modelos)
K = 5                        # fragmentos que se entregan al generador
CONFIG = json.loads((BASE / "rag" / "config.json").read_text(encoding="utf-8"))

## 3. Corpus e índice

In [ ]:
import pandas as pd
docs = construir_corpus(BASE, CONFIG)
trozos = trocear(docs)
resumen = pd.DataFrame(trozos).groupby("doc").agg(fragmentos=("id", "count"), secciones=("seccion", "nunique")).sort_values("fragmentos", ascending=False)
display(resumen)
print(f"{len(docs)} documentos · {len(trozos)} fragmentos")
buscador = Buscador(trozos, usar_embeddings=USAR_EMBEDDINGS)

## 4. Evaluación de la recuperación
Cada pregunta de control tiene una cadena que debe aparecer en algún fragmento recuperado. *Acierto@k*: proporción de preguntas con un fragmento correcto entre los k primeros; *MRR*: promedio de 1/posición del primer acierto.

In [ ]:
filas, metricas = evaluar(buscador, CONFIG["preguntas"], k=K)
display(pd.DataFrame(filas))
print({k: round(v, 3) for k, v in metricas.items()})
(BASE / "rag" / "evaluacion.json").write_text(json.dumps({"metricas": metricas, "detalle": filas}, ensure_ascii=False, indent=1, default=str), encoding="utf-8")

## 5. Chat

In [ ]:
generador = Generador(MODO_GENERADOR)
import html, ipywidgets as w
from IPython.display import display, HTML

def preguntar(pregunta, k=K):
    fragmentos = buscador.buscar(pregunta, k=k)
    respuesta = generador.responder(pregunta, fragmentos)
    fuentes = "".join(f"<li><b>[{i + 1}]</b> {html.escape(f['doc'])} · {html.escape(str(f['seccion']))}{' · p. ' + str(f['pagina']) if f['pagina'] else ''}"
                      f"<div style='color:#666;font-size:12px'>{html.escape(f['texto'][:320])}…</div></li>" for i, f in enumerate(fragmentos))
    return respuesta, fuentes

EJEMPLOS = ["¿El ingreso del café depende más de cantidades o de precios?", "¿Por qué la elasticidad del café es negativa?", "¿Qué muestra el panel espacial SAR?", "Are the Pasto and Ipiales potato markets integrated?", "¿Cómo se trató el cambio metodológico de la EVA?", "¿Qué recomienda el artículo para la política agropecuaria?"]
caja = w.Textarea(placeholder="Escriba su pregunta…", layout=w.Layout(width="100%", height="70px"))
ejemplo = w.Dropdown(options=["(preguntas de ejemplo)"] + EJEMPLOS, layout=w.Layout(width="100%"))
boton = w.Button(description="Preguntar", button_style="warning", icon="search")
limpiar = w.Button(description="Limpiar")
salida = w.Output()
historial = []

def mostrar():
    with salida:
        salida.clear_output()
        for q, r, f in reversed(historial):
            display(HTML(f"<div style='border:1px solid #e4e2dc;border-radius:10px;padding:12px;margin:8px 0;font-family:sans-serif'>"
                         f"<div style='color:#eb6834;font-weight:600'>{html.escape(q)}</div>"
                         f"<div style='margin:8px 0;white-space:pre-wrap'>{html.escape(r)}</div>"
                         f"<details><summary style='cursor:pointer;color:#2a78d6'>Fragmentos citados</summary><ol style='list-style:none;padding-left:0'>{f}</ol></details></div>"))

def al_preguntar(_):
    q = caja.value.strip()
    if not q:
        return
    boton.disabled, boton.description = True, "Buscando…"
    try:
        r, f = preguntar(q)
        historial.append((q, r, f))
        mostrar()
    finally:
        boton.disabled, boton.description = False, "Preguntar"

def al_elegir(cambio):
    if cambio["new"] != "(preguntas de ejemplo)":
        caja.value = cambio["new"]

boton.on_click(al_preguntar)
limpiar.on_click(lambda _: (historial.clear(), mostrar()))
ejemplo.observe(al_elegir, names="value")
display(w.VBox([w.HTML("<h3 style='font-family:sans-serif'>Pregunte sobre el proyecto</h3>"), ejemplo, caja, w.HBox([boton, limpiar]), salida]))

## 6. Uso sin interfaz
`preguntar("...")` devuelve la respuesta y las fuentes; útil para documentar consultas en el informe.

In [ ]:
r, _ = preguntar("¿El ingreso del café depende más de cantidades o de precios?")
print(r)